In [1]:
%cd ~/cdv
import os

# os.environ['CUDA_VISIBLE_DEVICES'] = ''

import numpy as np
import jax.numpy as jnp
import jax
import jax.random as jr
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
import rho_plus as rp
import wat
from eins import EinsOp
import treescope

# if os.environ['CUDA_VISIBLE_DEVICES'] == '':
#     jax.config.update('jax_default_device', jax.devices('cpu')[0])

is_dark = False
theme, cs = rp.mpl_setup(is_dark)
rp.plotly_setup(is_dark)

/home/nmiklaucic/miniconda3/envs/avid/lib/python3.12/site-packages/IPython/core/magics/osm.py:393: UserWarning: This is now an optional IPython functionality, using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})
/home/nmiklaucic/miniconda3/envs/avid/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/home/nmiklaucic/cdv


In [2]:
from pathlib import Path
import pyrallis
from facet.config import MainConfig
import orbax.checkpoint as ocp

from facet.training_state import TrainingRun
from facet.checkpointing import best_ckpt

# run_dir = Path('logs') / '08-31-22-23_488'
run_dir = Path('logs') / 'enb-199'

with open(run_dir / 'config.toml') as conf_file:
    config = pyrallis.cfgparsing.load(MainConfig, conf_file)

facet_model = config.build_regressor()

ckpt = best_ckpt(run_dir)
# ckpt = jax.tree.map(lambda x: x if isinstance(x, (float, int)) else x.astype(jnp.float32), ckpt)
params = ckpt['state']['params']
ema_params = ckpt['state']['opt_state'][-1]['ema']['params']

In [37]:
from facet.utils import load_pytree

with open('configs/sevennet.toml') as f:
    sevennet_conf = pyrallis.cfgparsing.load(MainConfig, f)

sevennet_mod = sevennet_conf.build_regressor()
sevennet_params = jax.tree.map(jnp.array, load_pytree('precomputed/sevennet.ckpt'))

In [38]:
mods = {
    'facet': facet_model,
    'sevennet': sevennet_mod
}

params = {
    'facet': {'params': ema_params},
    'sevennet': sevennet_params
}

In [39]:
from facet.data.databatch import CrystalGraphs
from facet.data.dataset import load_file


cgs = []
for i in range(1):
    cgs.append(load_file(sevennet_conf, group_num=15, file_num=i))

cg: CrystalGraphs = sum(cgs[1:], start=cgs[0])

In [40]:
from facet.layers import Context


def apply_mod(name):
    return lambda cg: mods[name].apply(params[name], ctx=Context(training=False), cg=cg)

In [42]:
funcs = {name: jax.jit(apply_mod(name)) for name in params}
yhats = {name: func(cg) for name, func in funcs.items()}

In [44]:
%%timeit
yhats['sevennet'] = funcs['sevennet'](cg).block_until_ready()

36.8 ms ± 419 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [45]:
%%timeit
yhats['facet'] = funcs['facet'](cg).block_until_ready()

20.9 ms ± 422 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [47]:
from facet.utils import debug_structure


debug_structure(params['sevennet']);

arg0 >>> params
├── edge_embedding >>> basis >>> freq
│   └── f32[8]
├── head
│   ├── Dense_0 >>> kernel
│   │   └── f32[128, 64]
│   └── Dense_1 >>> kernel
│       └── f32[64, 1]
├── mace
│   ├── layer_0 >>> interaction
│   │   ├── SimpleInteraction_0
│   │   │   ├── SevenNetConv_0 >>> LazyInMLP_0
│   │   │   │   ├── Dense_0
│   │   │   │   │   └── {...}
│   │   │   │   ├── Dense_1
│   │   │   │   │   └── {...}
│   │   │   │   └── Dense_2
│   │   │   │       └── {...}
│   │   │   ├── linear_intro >>> w[0,0] 128x0e,128x0e
│   │   │   │   └── f32[128, 128]
│   │   │   └── linear_outro
│   │   │       ├── w[0,0] 128x0e,224x0e
│   │   │       │   └── f32[128, 224]
│   │   │       ├── w[1,1] 128x1e,64x1e
│   │   │       │   └── f32[128, 64]
│   │   │       └── w[2,2] 128x2e,32x2e
│   │   │           └── f32[128, 32]
│   │   └── resid_adapter >>> w[0,0] 128x0e,224x0e
│   │       └── f32[128, 224]
│   ├── layer_1 >>> interaction
│   │   ├── SimpleInteraction_0
│   │   │   ├── SevenNetConv_0 >>> LazyInMLP_0
│   │   │   │   ├── Dense_0
│   │   │   │   │   └── {...}
│   │   │   │   ├── Dense_1
│   │   │   │   │   └── {...}
│   │   │   │   └── Dense_2
│   │   │   │       └── {...}
│   │   │   ├── linear_intro
│   │   │   │   ├── w[0,0] 128x0e,128x0e
│   │   │   │   │   └── f32[128, 128]
│   │   │   │   ├── w[1,1] 64x1e,64x1e
│   │   │   │   │   └── f32[64, 64]
│   │   │   │   └── w[2,2] 32x2e,32x2e
│   │   │   │       └── f32[32, 32]
│   │   │   └── linear_outro
│   │   │       ├── w[0,0] 224x0e,224x0e
│   │   │       │   └── f32[224, 224]
│   │   │       ├── w[1,1] 384x1e,64x1e
│   │   │       │   └── f32[384, 64]
│   │   │       └── w[2,2] 352x2e,32x2e
│   │   │           └── f32[352, 32]
│   │   └── resid_adapter
│   │       ├── w[0,0] 128x0e,224x0e
│   │       │   └── f32[128, 224]
│   │       ├── w[1,1] 64x1e,64x1e
│   │       │   └── f32[64, 64]
│   │       └── w[2,2] 32x2e,32x2e
│   │           └── f32[32, 32]
│   ├── layer_2 >>> interaction
│   │   ├── SimpleInteraction_0
│   │   │   ├── SevenNetConv_0 >>> LazyInMLP_0
│   │   │   │   ├── Dense_0
│   │   │   │   │   └── {...}
│   │   │   │   ├── Dense_1
│   │   │   │   │   └── {...}
│   │   │   │   └── Dense_2
│   │   │   │       └── {...}
│   │   │   ├── linear_intro
│   │   │   │   ├── w[0,0] 128x0e,128x0e
│   │   │   │   │   └── f32[128, 128]
│   │   │   │   ├── w[1,1] 64x1e,64x1e
│   │   │   │   │   └── f32[64, 64]
│   │   │   │   └── w[2,2] 32x2e,32x2e
│   │   │   │       └── f32[32, 32]
│   │   │   └── linear_outro
│   │   │       ├── w[0,0] 224x0e,224x0e
│   │   │       │   └── f32[224, 224]
│   │   │       ├── w[1,1] 384x1e,64x1e
│   │   │       │   └── f32[384, 64]
│   │   │       └── w[2,2] 352x2e,32x2e
│   │   │           └── f32[352, 32]
│   │   └── resid_adapter
│   │       ├── w[0,0] 128x0e,224x0e
│   │       │   └── f32[128, 224]
│   │       ├── w[1,1] 64x1e,64x1e
│   │       │   └── f32[64, 64]
│   │       └── w[2,2] 32x2e,32x2e
│   │           └── f32[32, 32]
│   ├── layer_3 >>> interaction
│   │   ├── SimpleInteraction_0
│   │   │   ├── SevenNetConv_0 >>> LazyInMLP_0
│   │   │   │   ├── Dense_0
│   │   │   │   │   └── {...}
│   │   │   │   ├── Dense_1
│   │   │   │   │   └── {...}
│   │   │   │   └── Dense_2
│   │   │   │       └── {...}
│   │   │   ├── linear_intro
│   │   │   │   ├── w[0,0] 128x0e,128x0e
│   │   │   │   │   └── f32[128, 128]
│   │   │   │   ├── w[1,1] 64x1e,64x1e
│   │   │   │   │   └── f32[64, 64]
│   │   │   │   └── w[2,2] 32x2e,32x2e
│   │   │   │       └── f32[32, 32]
│   │   │   └── linear_outro
│   │   │       ├── w[0,0] 224x0e,224x0e
│   │   │       │   └── f32[224, 224]
│   │   │       ├── w[1,1] 384x1e,64x1e
│   │   │       │   └── f32[384, 64]
│   │   │       └── w[2,2] 352x2e,32x2e
│   │   │           └── f32[352, 32]
│   │   └── resid_adapter
│   │       ├── w[0,0] 128x0e,224x0e
│   │       │   └── f32[128, 224]
│   │       ├── w[1,1] 64x1e,64x1e
│   │       │   └── f32[64, 64]
│   │       └── w[2,2] 32x2e,32x2e
│   │           └──